# CNN 图像分类：Digits 实验

## 本节目标

本节在 digits 数据集上训练小型 CNN，形成完整图像分类闭环。

完成本节后，你应该能够：

- 说清本节主题解决了什么问题。
- 读懂核心 PyTorch API 的输入输出。
- 运行最小代码示例并解释结果。
- 修改实验参数并观察变化。

## 背景问题

本节关注的问题是：本节在 digits 数据集上训练小型 CNN，形成完整图像分类闭环。

学习时建议先运行代码，再回头看解释。对初学者来说，能观察 shape、loss、accuracy 或可视化结果，比只记概念更重要。

## 核心概念

- 输入和输出的 shape 必须明确。
- loss 用来描述模型当前预测和目标之间的差距。
- 优化器根据梯度更新模型参数。
- 实验观察需要结合曲线、数值和样本可视化。
- 调试时优先检查 shape、dtype、学习率和训练/评估模式。

这里的关键不是堆公式，而是把概念落实到可运行代码。

## 最小代码示例：加载数据

这个代码块用于观察 `加载数据`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
x = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_train = torch.tensor(x_train).unsqueeze(1)
x_test = torch.tensor(x_test).unsqueeze(1)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)
print(x_train.shape, y_train.shape)

## 最小代码示例：定义模型并训练

这个代码块用于观察 `定义模型并训练`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32), nn.ReLU(),
            nn.Linear(32, 10),
        )
    def forward(self, x):
        return self.net(x)

loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
model = TinyCNN()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []
for epoch in range(25):
    total = 0
    for bx, by in loader:
        loss = loss_fn(model(bx), by)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * bx.size(0)
    losses.append(total / len(loader.dataset))

with torch.no_grad():
    pred = model(x_test).argmax(1)
    acc = (pred == y_test).float().mean().item()
print("test acc:", acc)

## 完整实验：结果展示

这个代码块用于观察 `结果展示`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i, 0], cmap="gray")
    plt.title(f"p:{pred[i].item()} y:{y_test[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("CNN loss")
plt.grid(alpha=0.3)
plt.show()

## 实验观察

小型 CNN 能在 digits 上快速取得较好结果。错误样本通常出现在形状相近的数字之间。

从运行结果可以看到，代码输出不只是“能跑”，还可以帮助判断模型是否在按预期学习。建议把观察写进实验记录，而不是只保留最终准确率。

## 常见错误

- shape 不匹配：先打印输入、输出和标签 shape。
- dtype 不正确：分类标签通常是 `torch.long`，回归目标通常是 float。
- 忘记 `optimizer.zero_grad()`：梯度会累积。
- 学习率不合理：过大震荡，过小收敛慢。
- 评估阶段忘记 `model.eval()` 或 `torch.no_grad()`。

调试时可以先缩小数据集，确认模型能否在小样本上过拟合。

## 概念问答

**Q：为什么要从最小代码示例开始？**  
A：最小示例能隔离核心概念，降低同时排查多个问题的难度。

**Q：为什么每个实验都要看曲线或可视化？**  
A：单个最终数值不一定能解释训练过程，曲线和样本结果更容易暴露问题。

**Q：如果代码能运行但结果不好，先查什么？**  
A：优先检查数据、shape、label dtype、loss 使用方式和学习率。

## 延伸练习

- 加入更多卷积通道。
- 对比 MLP 和 CNN 在同一数据上的表现。

这些练习不需要一次完成。建议每次只改一个变量，并记录改动前后的结果。

## 小结

本节把一个深度学习概念落到了可运行的 PyTorch 实验中。继续学习下一节前，建议确认自己能解释：输入是什么、模型做了什么、loss 如何计算、结果是否符合预期。